# Silver Layer

## Environment

In [0]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    DoubleType,
    LongType,
    ByteType,
)
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from functools import reduce

In [0]:
catalog_name = "databricks-repo"
schema_name = "silver"
volume_name = "raw_enem"

volume_base_path = f"/Volumes/{catalog_name}/{schema_name}/{volume_name}"

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{catalog_name}`.`{schema_name}`")

In [0]:
def convert_columns_with_schema(
    df: DataFrame, st: StructType
) -> DataFrame:
    """
    Converts columns to types by using the schema definition
    """
    return df.select(
        *[
            F.col(field.name).try_cast(field.dataType.simpleString())
            for field in st.fields
        ]
    )
class FeatureTables:
    def __init__(self, df: DataFrame):
        self._df = df

    def create_feature_table(self, table_name: str):
        """
        Creates a feature table in the catalog and schema specified in the notebook's configuration.
        """
        return self._df.filter(F.col("nome") == table_name).select(
            F.col("id_categoria").cast(IntegerType()),
            F.col("valor_categoria").cast(StringType()),
        )

## Table Preview

In [0]:
schema_participantes = StructType(
    [
        StructField("NU_INSCRICAO", LongType(), False),
        StructField("NU_ANO", IntegerType(), False),
        StructField("TP_FAIXA_ETARIA", IntegerType(), False),
        StructField("TP_SEXO", StringType(), False),
        StructField("TP_ESTADO_CIVIL", IntegerType(), False),
        StructField("TP_COR_RACA", IntegerType(), False),
        StructField("TP_NACIONALIDADE", IntegerType(), False),
        StructField("TP_ST_CONCLUSAO", IntegerType(), False),
        StructField("TP_ANO_CONCLUIU", IntegerType(), False),
        StructField("TP_ENSINO", IntegerType(), True),
        StructField("IN_TREINEIRO", IntegerType(), False),
        StructField("CO_MUNICIPIO_PROVA", IntegerType(), False),
        StructField("NO_MUNICIPIO_PROVA", StringType(), False),
        StructField("CO_UF_PROVA", IntegerType(), False),
        StructField("SG_UF_PROVA", StringType(), False),
        StructField("Q001", StringType(), False),
        StructField("Q002", StringType(), False),
        StructField("Q003", StringType(), False),
        StructField("Q004", StringType(), False),
        StructField("Q005", StringType(), False),
        StructField("Q006", StringType(), False),
        StructField("Q007", StringType(), False),
        StructField("Q008", StringType(), False),
        StructField("Q009", StringType(), False),
        StructField("Q010", StringType(), False),
        StructField("Q011", StringType(), False),
        StructField("Q012", StringType(), False),
        StructField("Q013", StringType(), False),
        StructField("Q014", StringType(), False),
        StructField("Q015", StringType(), False),
        StructField("Q016", StringType(), False),
        StructField("Q017", StringType(), False),
        StructField("Q018", StringType(), False),
        StructField("Q019", StringType(), False),
        StructField("Q020", StringType(), False),
        StructField("Q021", StringType(), False),
        StructField("Q022", StringType(), False),
        StructField("Q023", StringType(), False),
    ]
)
df_participantes = spark.read.table(f"`{catalog_name}`.bronze.participantes")
df_participantes = convert_columns_with_schema(
    df_participantes.filter(F.col("NU_INSCRICAO").isNotNull()), schema_participantes
)

df_participantes.show(10)

In [0]:
schema_resultados = StructType(
    [
        StructField("NU_SEQUENCIAL", StringType()),
        StructField("NU_ANO", IntegerType()),
        StructField("CO_ESCOLA", IntegerType(), True),
        StructField("CO_MUNICIPIO_ESC", IntegerType()),
        StructField("NO_MUNICIPIO_ESC", StringType()),
        StructField("CO_UF_ESC", IntegerType()),
        StructField("SG_UF_ESC", StringType()),
        StructField("TP_DEPENDENCIA_ADM_ESC", IntegerType()),
        StructField("TP_LOCALIZACAO_ESC", IntegerType()),
        StructField("TP_SIT_FUNC_ESC", IntegerType()),
        StructField("CO_MUNICIPIO_PROVA", IntegerType()),
        StructField("NO_MUNICIPIO_PROVA", StringType()),
        StructField("CO_UF_PROVA", IntegerType()),
        StructField("SG_UF_PROVA", StringType()),
        StructField("TP_PRESENCA_CN", IntegerType()),
        StructField("TP_PRESENCA_CH", IntegerType()),
        StructField("TP_PRESENCA_LC", IntegerType()),
        StructField("TP_PRESENCA_MT", IntegerType()),
        StructField("CO_PROVA_CN", IntegerType()),
        StructField("CO_PROVA_CH", IntegerType()),
        StructField("CO_PROVA_LC", IntegerType()),
        StructField("CO_PROVA_MT", IntegerType()),
        StructField("NU_NOTA_CN", DoubleType()),
        StructField("NU_NOTA_CH", DoubleType()),
        StructField("NU_NOTA_LC", DoubleType()),
        StructField("NU_NOTA_MT", DoubleType()),
        StructField("TX_RESPOSTAS_CN", StringType()),
        StructField("TX_RESPOSTAS_CH", StringType()),
        StructField("TX_RESPOSTAS_LC", StringType()),
        StructField("TX_RESPOSTAS_MT", StringType()),
        StructField("TP_LINGUA", ByteType()),
        StructField("TX_GABARITO_CN", StringType()),
        StructField("TX_GABARITO_CH", StringType()),
        StructField("TX_GABARITO_LC", StringType()),
        StructField("TX_GABARITO_MT", StringType()),
        StructField("TP_STATUS_REDACAO", ByteType()),
        StructField("NU_NOTA_COMP1", IntegerType()),
        StructField("NU_NOTA_COMP2", IntegerType()),
        StructField("NU_NOTA_COMP3", IntegerType()),
        StructField("NU_NOTA_COMP4", IntegerType()),
        StructField("NU_NOTA_COMP5", IntegerType()),
        StructField("NU_NOTA_REDACAO", IntegerType()),
    ]
)
df_resultados = spark.read.table(f"`{catalog_name}`.bronze.resultados")
df_resultados = convert_columns_with_schema(
    df_resultados.filter(F.col("NU_SEQUENCIAL").isNotNull()), schema_resultados
)
df_resultados.show(10)

## Transformation

In [0]:
schema_itens_prova = StructType(
    [
        StructField("CO_POSICAO", IntegerType()),
        StructField("SG_AREA", StringType()),
        StructField("CO_ITEM", IntegerType()),
        StructField("TX_GABARITO", StringType()),
        StructField("CO_HABILIDADE", IntegerType()),
        StructField("IN_ITEM_ABAN", ByteType()),
        StructField("TX_MOTIVO_ABAN", StringType()),
        StructField("NU_PARAM_A", DoubleType()),
        StructField("NU_PARAM_B", DoubleType()),
        StructField("NU_PARAM_C", DoubleType()),
        StructField("TX_COR", StringType()),
        StructField("CO_PROVA", IntegerType()),
        StructField("TP_LINGUA", ByteType()),
        StructField("IN_ITEM_ADAPTADO", ByteType()),
    ]
)
df_itens_prova = spark.read.table(f"`{catalog_name}`.bronze.itens_prova")
df_itens_prova = convert_columns_with_schema(df_itens_prova, schema_itens_prova)

df_itens_prova.show(10)

In [0]:
dicionario_dados = (
    spark.read.table("`databricks-repo`.bronze.dicionario_dados")
    .select(
        F.col("_c0").alias("nome"),
        F.col("_c1").alias("descricao"),
        F.col("_c2").alias("id_categoria"),
        F.col("_c3").alias("valor_categoria"),
    )
    .withColumn("index", F.monotonically_increasing_id())
    .filter(F.col("index") >= 4)
)

window_spec = Window.orderBy("index").rowsBetween(
    Window.unboundedPreceding, Window.currentRow
)

dicionario_dados = dicionario_dados.withColumn(
    "nome", F.last("nome", ignorenulls=True).over(window_spec)
).withColumn("descricao", F.last("descricao", ignorenulls=True).over(window_spec))

display(dicionario_dados)

## Features Tables

In [0]:
f_tables = FeatureTables(dicionario_dados)

features_tables = {
    "ft_faixa_etaria": f_tables.create_feature_table("TP_FAIXA_ETARIA"),
    "ft_estado_civil": f_tables.create_feature_table("TP_ESTADO_CIVIL"),
    "ft_cor_raca": f_tables.create_feature_table("TP_COR_RACA"),
    "ft_nacionalidade": f_tables.create_feature_table("TP_NACIONALIDADE"),
    "ft_conclusao": f_tables.create_feature_table("TP_ST_CONCLUSAO"),
    "ft_ensino": f_tables.create_feature_table("TP_ENSINO"),
}

for _name, _table in features_tables.items():
    _table.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(
        f"`{catalog_name}`.{schema_name}.{_name}"
    )

In [0]:
df_municipios = (
    df_resultados.select(
        F.col("CO_MUNICIPIO_PROVA").alias("id_muninicipio"),
        F.col("NO_MUNICIPIO_PROVA").alias("muninicipio"),
    )
    .distinct()
    .write.mode("overwrite")
    .saveAsTable(f"`{catalog_name}`.{schema_name}.municipio")
)

df_unidade_federativa = (
    df_resultados.select(
        F.col("CO_UF_PROVA").alias("id_uf"), F.col("SG_UF_PROVA").alias("uf_sigla")
    )
    .distinct()
    .write.mode("overwrite")
    .saveAsTable(f"`{catalog_name}`.{schema_name}.unidade_federativa")
)

In [0]:
escola = (
    df_resultados.select(
        F.col("CO_ESCOLA").alias("id_escola"),
        F.col("CO_MUNICIPIO_ESC").alias("id_municipio"),
        F.col("CO_UF_ESC").alias("id_uf"),
        F.col("TP_DEPENDENCIA_ADM_ESC").alias("tipo_escola"),
        F.col("TP_SIT_FUNC_ESC").alias("situacao_funcional"),
    )
    .distinct()
    .dropna(how="all")
)
escola.groupBy().agg(
    F.countDistinct("id_escola").alias("num_escolas"),
    F.countDistinct("id_municipio").alias("num_municipios"),
    F.countDistinct("id_uf").alias("num_ufs"),
    F.countDistinct("tipo_escola").alias("num_tipos_escola"),
    F.countDistinct("situacao_funcional").alias("num_situacao_funcional"),
    # Count Null values
    F.sum(F.col("id_escola").isNull().cast(IntegerType())).alias("num_escolas_nulls"),
    F.sum(F.col("id_municipio").isNull().cast(IntegerType())).alias("num_mun_nulls"),
    F.sum(F.col("id_uf").isNull().cast(IntegerType())).alias("num_ufs_nulls"),
    F.sum(F.col("tipo_escola").isNull().cast(IntegerType())).alias("num_tipo_esc_null"),
    F.sum(F.col("situacao_funcional").isNull().cast(IntegerType())).alias(
        "num_sit_funcional_null"
    ),
).show()

escola.write.mode("overwrite").saveAsTable(f"`{catalog_name}`.{schema_name}.escola")

In [0]:
local_aplicacao = df_resultados.select(
    F.col("CO_MUNICIPIO_PROVA").alias("id_municipio"),
    F.col("NU_SEQUENCIAL").alias("id_aplicacao"),
    F.col("CO_UF_PROVA").alias("id_uf"),
).distinct()

local_aplicacao.write.mode("overwrite").saveAsTable(f"`{catalog_name}`.{schema_name}.local_aplicacao")

In [0]:
situacao_prova = df_resultados.select(
    F.col("NU_SEQUENCIAL").alias("id_aplicacao"),
    F.col("TP_PRESENCA_CN").alias("id_presenca_ciencias_natureza"),
    F.col("TP_PRESENCA_CH").alias("id_presenca_ciencias_humanas"),
    F.col("TP_PRESENCA_LC").alias("id_presenca_linguagens"),
    F.col("TP_PRESENCA_MT").alias("id_presenca_matematica"),
).distinct()

In [0]:
disciplinas = ["CN", "CH", "LC", "MT"]
aplicacao = (
    df_resultados.unpivot(
        "NU_SEQUENCIAL", [f"CO_PROVA_{i}" for i in disciplinas], "c", "id_caderno"
    )
    .select(F.col("NU_SEQUENCIAL").alias("id_aplicacao"), "id_caderno")
    .filter(F.col("id_caderno").isNotNull())
    .distinct()
)

aplicacao.orderBy("id_aplicacao").show()

## Competencias

### Linguagens

#### Aplicar as tecnologias da comunicação e da informação na escola, no trabalho e em outros contextos relevantes para sua vida.

H1. Identificar as diferentes linguagens e seus recursos expressivos como elementos de caracterização dos sistemas de comunicação.

H2. Recorrer aos conhecimentos sobre as linguagens dos sistemas de comunicação e informação para resolver problemas sociais.

H3. Relacionar informações geradas nos sistemas de comunicação e informação, considerando a função social desses sistemas.

H4. Reconhecer posições críticas aos usos sociais que são feitos das linguagens e dos sistemas de comunicação e informação.

#### Conhecer e usar língua(s) estrangeira(s) moderna(s) como instrumento de acesso a informações e a outras culturas e grupos sociais.

H5. Associar vocábulos e expressões de um texto em LEM ao seu tema.

H6. Utilizar os conhecimentos da LEM e de seus mecanismos como meio de ampliar as possibilidades de acesso a informações, tecnologias e culturas.

H7. Relacionar um texto em LEM, as estruturas linguísticas, sua função e seu uso social.

H8. Reconhecer a importância da produção cultural em LEM como representação da diversidade cultural e linguística.

#### Compreender e usar a linguagem corporal como relevante para a própria vida, integradora social e formadora da identidade.

H9. Reconhecer as manifestações corporais de movimento como originárias de necessidades cotidianas de um grupo social.

H10. Reconhecer a necessidade de transformação de hábitos corporais em função das necessidades cinestésicas.

H11. Reconhecer a linguagem corporal como meio de interação social, considerando os limites de desempenho e as alternativas de adaptação para diferentes indivíduos.

#### Compreender a arte como saber cultural e estético gerador de significação e integrador da organização do mundo e da própria identidade.

H12. Reconhecer diferentes funções da arte, do trabalho da produção dos artistas em seus meios culturais.

H13. Analisar as diversas produções artísticas como meio de explicar diferentes culturas, padrões de beleza e preconceitos.

H14. Reconhecer o valor da diversidade artística e das inter-relações de elementos que se apresentam nas manifestações de vários grupos sociais e étnicos.

#### Analisar, interpretar e aplicar recursos expressivos das linguagens, relacionando textos com seus contextos, mediante a natureza, função, organização, estrutura das manifestações, de acordo com as condições de produção e recepção.

H15. Estabelecer relações entre o texto literário e o momento de sua produção, situando aspectos do contexto histórico, social e político.

H16. Relacionar informações sobre concepções artísticas e procedimentos de construção do texto literário.

H17. Reconhecer a presença de valores sociais e humanos atualizáveis e permanentes no patrimônio literário nacional.

#### Compreender e usar os sistemas simbólicos das diferentes linguagens como meios de organização cognitiva da realidade pela constituição de significados, expressão, comunicação e informação.

H18. Identificar os elementos que concorrem para a progressão temática e para a organização e estruturação de textos de diferentes gêneros e tipos.

H19. Analisar a função da linguagem predominante nos textos em situações específicas de interlocução.

H20. Reconhecer a importância do patrimônio linguístico para a preservação da memória e da identidade nacional.

#### Confrontar opiniões e pontos de vista sobre as diferentes linguagens e suas manifestações específicas.

H21. Reconhecer em textos de diferentes gêneros, recursos verbais e não-verbais utilizados com a finalidade de criar e mudar comportamentos e hábitos.

H22. Relacionar, em diferentes textos, opiniões, temas, assuntos e recursos linguísticos.

H23. Inferir em um texto quais são os objetivos de seu produtor e quem é seu público alvo, pela análise dos procedimentos argumentativos utilizados.

H24. Reconhecer no texto estratégias argumentativas empregadas para o convencimento do público, tais como a intimidação, sedução, comoção, chantagem, entre outras.

#### Compreender e usar a língua portuguesa como língua materna, geradora de significação e integradora da organização do mundo e da própria identidade.

H25. Identificar, em textos de diferentes gêneros, as marcas linguísticas que singularizam as variedades linguísticas sociais, regionais e de registro.

H26. Relacionar as variedades linguísticas a situações específicas de uso social.

H27. Reconhecer os usos da norma padrão da língua portuguesa nas diferentes situações de comunicação.

#### Entender os princípios, a natureza, a função e o impacto das tecnologias da comunicação e da informação na sua vida pessoal e social, no desenvolvimento do conhecimento, associando-o aos conhecimentos científicos, às linguagens que lhes dão suporte, às demais tecnologias, aos processos de produção e aos problemas que se propõem solucionar.

H28. Reconhecer a função e o impacto social das diferentes tecnologias da comunicação e informação.

H29. Identificar pela análise de suas linguagens, as tecnologias da comunicação e informação.

H30. Relacionar as tecnologias de comunicação e informação ao desenvolvimento das sociedades e ao conhecimento que elas produzem.

---

### Matemática e suas Tecnologias


#### Construir significados para os números naturais, inteiros, racionais e reais.

H1. Reconhecer, no contexto social, diferentes significados e representações dos números e operações - naturais, inteiros, racionais ou reais.

H2. Identificar padrões numéricos ou princípios de contagem.

H3. Resolver situação-problema envolvendo conhecimentos numéricos.

H4. Avaliar a razoabilidade de um resultado numérico na construção de argumentos sobre afirmações quantitativas.

H5. Avaliar propostas de intervenção na realidade utilizando conhecimentos numéricos.

#### Utilizar o conhecimento geométrico para realizar a leitura e a representação da realidade e agir sobre ela.

H6. Interpretar a localização e a movimentação de pessoas/objetos no espaço tridimensional e sua representação no espaço bidimensional.

H7. Identificar características de figuras planas ou espaciais.

H8. Resolver situação-problema que envolva conhecimentos geométricos de espaço e forma.

H9. Utilizar conhecimentos geométricos de espaço e forma na seleção de argumentos propostos como solução de problemas do cotidiano.

#### Construir noções de grandezas e medidas para a compreensão da realidade e a solução de problemas do cotidiano.

H10. Identificar relações entre grandezas e unidades de medida.

H11. Utilizar a noção de escalas na leitura de representação de situação do cotidiano.

H12. Resolver situação-problema que envolva medidas de grandezas.

H13. Avaliar o resultado de uma medição na construção de um argumento consistente.

H14. Avaliar proposta de intervenção na realidade utilizando conhecimentos geométricos relacionados a grandezas e medidas.

#### Construir noções de variação de grandezas para a compreensão da realidade e a solução de problemas do cotidiano.

H15. Identificar a relação de dependência entre grandezas.

H16. Resolver situação-problema envolvendo a variação de grandezas, direta ou inversamente proporcionais.

H17. Analisar informações envolvendo a variação de grandezas como recurso para a construção de argumentação.

H18. Avaliar propostas de intervenção na realidade envolvendo variação de grandezas.

#### Modelar e resolver problemas que envolvem variáveis socioeconômicas ou técnico-científicas, usando representações algébricas.

H19. Identificar representações algébricas que expressem a relação entre grandezas.

H20. Interpretar gráfico cartesiano que represente relações entre grandezas.

H21. Resolver situação-problema cuja modelagem envolva conhecimentos algébricos.

H22. Utilizar conhecimentos algébricos/geométricos como recurso para a construção de argumentação.

H23. Avaliar propostas de intervenção na realidade utilizando conhecimentos algébricos.

#### Interpretar informações de natureza científica e social obtidas da leitura de gráficos e tabelas, realizando previsão de tendência, extrapolação, interpolação e interpretação.

H24. Utilizar informações expressas em gráficos ou tabelas para fazer inferências.

H25. Resolver problema com dados apresentados em tabelas ou gráficos.

H26. Analisar informações expressas em gráficos ou tabelas como recurso para a construção de argumentos.

#### Compreender o caráter aleatório e não-determinístico dos fenômenos naturais e sociais e utilizar instrumentos adequados para medidas, determinação de amostras e cálculos de probabilidade para interpretar informações de variáveis apresentadas em uma distribuição estatística.

H27. Calcular medidas de tendência central ou de dispersão de um conjunto de dados expressos em uma tabela de frequências de dados agrupados (não em classes) ou em gráficos.

H28. Resolver situação-problema que envolva conhecimentos de estatística e probabilidade.

H29. Utilizar conhecimentos de estatística e probabilidade como recurso para a construção de argumentação.

H30. Avaliar propostas de intervenção na realidade utilizando conhecimentos de estatística e probabilidade.

---

### Ciências da Natureza e suas Tecnologias


#### Compreender as ciências naturais e as tecnologias a elas associadas como construções humanas, percebendo seus papéis nos processos de produção e no desenvolvimento econômico e social da humanidade.

H1. Reconhecer características ou propriedades de fenômenos ondulatórios ou oscilatórios, relacionando-os a seus usos em diferentes contextos.

H2. Associar a solução de problemas de comunicação, transporte, saúde ou outro, com o correspondente desenvolvimento científico e tecnológico.

H3. Confrontar interpretações científicas com interpretações baseadas no senso comum, ao longo do tempo ou em diferentes culturas.

H4. Avaliar propostas de intervenção no ambiente, considerando a qualidade da vida humana ou medidas de conservação, recuperação ou utilização sustentável da biodiversidade.

#### Identificar a presença e aplicar as tecnologias associadas às ciências naturais em diferentes contextos.

H5. Dimensionar circuitos ou dispositivos elétricos de uso cotidiano.

H6. Relacionar informações para compreender manuais de instalação ou utilização de aparelhos, ou sistemas tecnológicos de uso comum.

H7. Selecionar testes de controle, parâmetros ou critérios para a comparação de materiais e produtos, tendo em vista a defesa do consumidor, a saúde do trabalhador ou a qualidade de vida.

#### Associar intervenções que resultam em degradação ou conservação ambiental a processos produtivos e sociais e a instrumentos ou ações científico-tecnológicos.

H8. Identificar etapas em processos de obtenção, transformação, utilização ou reciclagem de recursos naturais, energéticos ou matérias-primas, considerando processos biológicos, químicos ou físicos neles envolvidos.

H9. Compreender a importância dos ciclos biogeoquímicos ou do fluxo energia para a vida, ou da ação de agentes ou fenômenos que podem causar alterações nesses processos.

H10. Analisar perturbações ambientais, identificando fontes, transporte e(ou) destino dos poluentes ou prevendo efeitos em sistemas naturais, produtivos ou sociais.

H11. Reconhecer benefícios, limitações e aspectos éticos da biotecnologia, considerando estruturas e processos biológicos envolvidos em produtos biotecnológicos.

H12. Avaliar impactos em ambientes naturais decorrentes de atividades sociais ou econômicas, considerando interesses contraditórios.

#### Compreender interações entre organismos e ambiente, em particular aquelas relacionadas à saúde humana, relacionando conhecimentos científicos, aspectos culturais e características individuais.

H13. Reconhecer mecanismos de transmissão da vida, prevendo ou explicando a manifestação de características dos seres vivos.

H14. Identificar padrões em fenômenos e processos vitais dos organismos, como manutenção do equilíbrio interno, defesa, relações com o ambiente, sexualidade, entre outros.

H15. Interpretar modelos e experimentos para explicar fenômenos ou processos biológicos em qualquer nível de organização dos sistemas biológicos.

H16. Compreender o papel da evolução na produção de padrões, processos biológicos ou na organização taxonômica dos seres vivos.

#### Entender métodos e procedimentos próprios das ciências naturais e aplicá-los em diferentes contextos.

H17. Relacionar informações apresentadas em diferentes formas de linguagem e representação usadas nas ciências físicas, químicas ou biológicas, como texto discursivo, gráficos, tabelas, relações matemáticas ou linguagem simbólica.

H18. Relacionar propriedades físicas, químicas ou biológicas de produtos, sistemas ou procedimentos tecnológicos às finalidades a que se destinam.

H19. Avaliar métodos, processos ou procedimentos das ciências naturais que contribuam para diagnosticar ou solucionar problemas de ordem social, econômica ou ambiental.

#### Apropriar-se de conhecimentos da física para, em situações problema, interpretar, avaliar ou planejar intervenções científico-tecnológicas.

H20. Caracterizar causas ou efeitos dos movimentos de partículas, substâncias, objetos ou corpos celestes.

H21. Utilizar leis físicas e (ou) químicas para interpretar processos naturais ou tecnológicos inseridos no contexto da termodinâmica e(ou) do eletromagnetismo.

H22. Compreender fenômenos decorrentes da interação entre a radiação e a matéria em suas manifestações em processos naturais ou tecnológicos, ou em suas implicações biológicas, sociais, econômicas ou ambientais.

H23. Avaliar possibilidades de geração, uso ou transformação de energia em ambientes específicos, considerando implicações éticas, ambientais, sociais e/ou econômicas.

#### Apropriar-se de conhecimentos da química para, em situações problema, interpretar, avaliar ou planejar intervenções científico-tecnológicas.

H24. Utilizar códigos e nomenclatura da química para caracterizar materiais, substâncias ou transformações químicas.

H25. Caracterizar materiais ou substâncias, identificando etapas, rendimentos ou implicações biológicas, sociais, econômicas ou ambientais de sua obtenção ou produção.

H26. Avaliar implicações sociais, ambientais e/ou econômicas na produção ou no consumo de recursos energéticos ou minerais, identificando transformações químicas ou de energia envolvidas nesses processos.

H27. Avaliar propostas de intervenção no meio ambiente aplicando conhecimentos químicos, observando riscos ou benefícios.

#### Apropriar-se de conhecimentos da biologia para, em situações problema, interpretar, avaliar ou planejar intervenções científico-tecnológicas.

H28. Associar características adaptativas dos organismos com seu modo de vida ou com seus limites de distribuição em diferentes ambientes, em especial em ambientes brasileiros.

H29. Interpretar experimentos ou técnicas que utilizam seres vivos, analisando implicações para o ambiente, a saúde, a produção de alimentos, matérias primas ou produtos industriais.

H30. Avaliar propostas de alcance individual ou coletivo, identificando aquelas que visam à preservação e a implementação da saúde individual, coletiva ou do ambiente.

---

### Ciências Humanas e suas Tecnologias


#### Compreender os elementos culturais que constituem as identidades.

H1. Interpretar historicamente e/ou geograficamente fontes documentais acerca de aspectos da cultura.

H2. Analisar a produção da memória pelas sociedades humanas.

H3. Associar as manifestações culturais do presente aos seus processos históricos.

H4. Comparar pontos de vista expressos em diferentes fontes sobre determinado aspecto da cultura.

H5. Identificar as manifestações ou representações da diversidade do patrimônio cultural e artístico em diferentes sociedades.

#### Compreender as transformações dos espaços geográficos como produto das relações socioeconômicas e culturais de poder.

H6. Interpretar diferentes representações gráficas e cartográficas dos espaços geográficos.

H7. Identificar os significados histórico-geográficos das relações de poder entre as nações.

H8. Analisar a ação dos estados nacionais no que se refere à dinâmica dos fluxos populacionais e no enfrentamento de problemas de ordem econômico-social.

H9. Comparar o significado histórico-geográfico das organizações políticas e socioeconômicas em escala local, regional ou mundial.

H10. Reconhecer a dinâmica da organização dos movimentos sociais e a importância da participação da coletividade na transformação da realidade histórico-geográfica.

#### Compreender a produção e o papel histórico das instituições sociais, políticas e econômicas, associando-as aos diferentes grupos, conflitos e movimentos sociais.

H11. Identificar registros de práticas de grupos sociais no tempo e no espaço.

H12. Analisar o papel da justiça como instituição na organização das sociedades.

H13. Analisar a atuação dos movimentos sociais que contribuíram para mudanças ou rupturas em processos de disputa pelo poder.

H14. Comparar diferentes pontos de vista, presentes em textos analíticos e interpretativos, sobre situação ou fatos de 
natureza histórico-geográfica acerca das instituições sociais, políticas e econômicas.

H15. Avaliar criticamente conflitos culturais, sociais, políticos, econômicos ou ambientais ao longo da história.

#### Entender as transformações técnicas e tecnológicas e seu impacto nos processos de produção, no desenvolvimento do conhecimento e na vida social.

H16. Identificar registros sobre o papel das técnicas e tecnologias na organização do trabalho e/ou da vida social.

H17. Analisar fatores que explicam o impacto das novas tecnologias no processo de territorialização da produção.

H18. Analisar diferentes processos de produção ou circulação de riquezas e suas implicações sócio-espaciais.

H19. Reconhecer as transformações técnicas e tecnológicas que determinam as várias formas de uso e apropriação dos espaços rural e urbano.

H20. Selecionar argumentos favoráveis ou contrários às modificações impostas pelas novas tecnologias à vida social e ao mundo do trabalho.

#### Utilizar os conhecimentos históricos para compreender e valorizar os fundamentos da cidadania e da democracia, favorecendo uma atuação consciente do indivíduo na sociedade.

H21. Identificar o papel dos meios de comunicação na construção da vida social.

H22. Analisar as lutas sociais e conquistas obtidas no que se refere às mudanças nas legislações ou nas políticas públicas.

H23. Analisar a importância dos valores éticos na estruturação política das sociedades.

H24. Relacionar cidadania e democracia na organização das sociedades.

H25. Identificar estratégias que promovam formas de inclusão social.

#### Compreender a sociedade e a natureza, reconhecendo suas interações no espaço em diferentes contextos históricos e geográficos.

H26. Identificar em fontes diversas o processo de ocupação dos meios físicos e as relações da vida humana com a paisagem.

H27. Analisar de maneira crítica as interações da sociedade com o meio físico, levando em consideração aspectos históricos e(ou) geográficos.

H28. Relacionar o uso das tecnologias com os impactos sócio-ambientais em diferentes contextos histórico-geográficos.

H29. Reconhecer a função dos recursos naturais na produção do espaço geográfico, relacionando-os com as mudanças provocadas pelas ações humanas.

H30. Avaliar as relações entre preservação e degradação da vida no planeta nas diferentes escalas.

## Itens Prova

In [0]:
df_itens_prova.show(10)

In [0]:
questoes = df_itens_prova.select(
    F.col("CO_ITEM").alias("id_questao"),
    F.col("CO_HABILIDADE").alias("id_habilidade"),
    F.col("TX_GABARITO").alias("alternativa_correta"),
    F.col("NU_PARAM_A").alias("stats_dominio_item"),
    F.col("NU_PARAM_B").alias("stats_dificuldade"),
    F.col("NU_PARAM_C").alias("stats_acerto_chute"),
).distinct()

questoes.show()


prova = df_itens_prova.select(
    F.col("TX_COR").alias("cor"),
    F.col("IN_ITEM_ADAPTADO").alias("is_adaptado"),
    F.col("SG_AREA").alias("area"),
    F.col("CO_PROVA").alias("id_prova"),
).distinct()

group_by_columns = ["CO_PROVA", "SG_AREA"]
display(
    df_itens_prova.groupBy(*group_by_columns).agg(
        *[
            F.countDistinct(i)
            for i in df_itens_prova.columns
            if i not in group_by_columns
        ]
    )
)

questoes_in_prova = df_itens_prova.select(
    F.col('CO_ITEM'),
    F.col('CO_PROVA')
    )

prova.write.mode("overwrite").saveAsTable(f"`{catalog_name}`.{schema_name}.prova")
questoes.write.mode("overwrite").saveAsTable(f"`{catalog_name}`.{schema_name}.questoes")